<a href="https://colab.research.google.com/github/HBoiarsky/MVA/blob/main/foundations_in_LLM/NLP_before_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Exercice 2.2. Measure the perplexity of the unigram and bigram models

In [ ]:
import math
from datasets import load_dataset
wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

# train sequence
train_sequence = []
for line in wikitext["train"]["text"]:
  train_sequence.extend(line.split())

# validation sequence
val_sequence = []
for line in wikitext["validation"]["text"]:
    val_sequence.extend(line.split())

# test sequence
test_sequence = []
for line in wikitext["test"]["text"]:
    test_sequence.extend(line.split())

# unigram count
unigram_count = {}
for a in train_sequence:
  if a not in unigram_count:
    unigram_count[a] = 0
  unigram_count[a] += 1

# perplexity unigram
N = sum(unigram_count.values())
probability_unigram = {}
for word, count in unigram_count.items():
  probability_unigram[word] = count / N

sum_log_proba = 0
for word in val_sequence:
  p = probability_unigram.get(word, 0)
  if p == 0:
    sum_log_proba = float("-inf")
    break
  sum_log_proba += math.log(p)

perplexity_unigram = (
    math.inf if sum_log_proba == float("-inf")
    else math.exp(-(1/(len(val_sequence))) * sum_log_proba))

# bigram count
bigram_count = {}
for a, b in zip(train_sequence, train_sequence[1:]):
  if a not in bigram_count:
    bigram_count[a] = {}
  if b not in bigram_count[a]:
    bigram_count[a][b] = 0
  bigram_count[a][b] += 1

# perplexity bigram
probability_bigram = {}
for a, b in zip(train_sequence, train_sequence[1:]):
  if a not in probability_bigram:
    probability_bigram[a] = {}
  if b not in probability_bigram[a]:
    probability_bigram[a][b] = 0
  probability_bigram[a][b] = bigram_count[a][b] / unigram_count[a]

sum_log_proba = 0
for word_previous, word in zip(val_sequence, val_sequence[1:]):
  p = probability_bigram.get(word_previous, {}).get(word, 0)
  if p == 0:
    sum_log_proba = float("-inf")
    break
  sum_log_proba += math.log(p)

perplexity_bigram = (
    math.inf if sum_log_proba == float("-inf")
    else math.exp(-(1/(len(val_sequence)-1)) * sum_log_proba))


In [ ]:
perplexity_bigram

inf

In [ ]:
perplexity_unigram

996.5031614048719

#Exercice 3.2. Classify the IMBD reviews with a bag of words

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")

def clean(texts):
    return [text.replace("<br />", " ") for text in texts]

# train, val and test sets
split = imdb["train"].train_test_split(test_size=5000, seed=0)
train_text = clean(split["train"]["text"])
val_text = clean(split["test"]["text"])
test_text = clean(imdb["test"]["text"])
train_label = split["train"]["label"]
val_label = split["test"]["label"]
test_label = imdb["test"]["label"]

# count the words of each review over the 10000 most frequent words
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=10000,
                             token_pattern=r"\b\w+\b")

X_train = vectorizer.fit_transform(train_text)
X_val = vectorizer.transform(val_text)
X_test = vectorizer.transform(test_text)

# logistic regression on X_train
from sklearn.linear_model import LogisticRegression

best_accuracy=0
for C in [0.01, 0.03, 0.1, 0.3, 1, 3, 10]:
  model = LogisticRegression(C= C, max_iter=1000)
  model.fit(X_train, train_label)
  accuracy = model.score(X_val, val_label)
  print(f"C = {C}: validation accuracy {accuracy:.3f}")
  if accuracy > best_accuracy:
    best_accuracy = accuracy
    best_model = model
    best_C = C

# test accuracy
test_accuracy = best_model.score(X_test, test_label)
print(f"\ntest accuracy, C = {best_C}: {test_accuracy:.3f}")

C = 0.01: validation accuracy 0.882
C = 0.03: validation accuracy 0.887
C = 0.1: validation accuracy 0.889
C = 0.3: validation accuracy 0.886
C = 1: validation accuracy 0.875
C = 3: validation accuracy 0.870
C = 10: validation accuracy 0.864

test accuracy, C = 0.1: 0.876


#Exercice 4.2. Train Bengio's neural language model

In [ ]:
from datasets import load_dataset

wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

# train, val and test sequences
train_sequence = []
for line in wikitext["train"]["text"]:
    train_sequence.extend(line.split())

val_sequence = []
for line in wikitext["validation"]["text"]:
    val_sequence.extend(line.split())

test_sequence = []
for line in wikitext["test"]["text"]:
    test_sequence.extend(line.split())

# give an index for each word of the vocabulary
vocabulary = sorted(set(train_sequence))
word_index = {word:i for i, word in enumerate(vocabulary)}

# turn a sequence into (context, next word) pair
import torch
context_size = 3

def make_pairs(sequence):
  indices = [word_index[word] for word in sequence]
  context = [indices[i - context_size:i]
             for i in range(context_size, len(indices))]
  target = indices[context_size:]
  return torch.tensor(context), torch.tensor(target)

train_context, train_target = make_pairs(train_sequence)
val_context, val_target = make_pairs(val_sequence)
test_context, test_target = make_pairs(test_sequence)

In [ ]:
# (1) Build the model: look up the embeddings of the 3 words of
# the context, concatenate them, then a hidden layer (tanh) and
# one score per word of the vocabulary. Start from small
# embeddings: with the default initialization, of size 1, the
# rare words would keep their random vectors.
from torch import nn

V = len(vocabulary)
D = 100   # size of the embeddings
H = 200   # size of the hidden layer
torch.manual_seed(0)
model = nn.Sequential(
    nn.Embedding(V, D),   # (batch, 3) -> (batch, 3, D)
    nn.Flatten(),         # concatenation: (batch, 3 * D)
    nn.Linear(context_size * D, H),
    nn.Tanh(),
    nn.Linear(H, V),      # one score per word
)
nn.init.normal_(model[0].weight, std=0.1)

# Run on the GPU if there is one
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)


# (2) The perplexity of a model on a set of pairs: the exponential
# of the average cross-entropy loss. F.cross_entropy applies the
# softmax to the scores, then computes the loss of section 2.2.
import math
import torch.nn.functional as F

def perplexity(model, contexts, targets):
    total_loss = 0
    with torch.no_grad():
        for start in range(0, len(targets), 10000):
            scores = model(contexts[start:start + 10000].to(device))
            batch_targets = targets[start:start + 10000].to(device)
            total_loss += F.cross_entropy(scores, batch_targets,
                                          reduction="sum").item()
    return math.exp(total_loss / len(targets))


# (3) Train with Adam on batches of 512 pairs. After each epoch,
# measure the validation perplexity, and keep a copy of the best
# model.
import copy

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 512
best_perplexity = math.inf
for epoch in range(4):
    order = torch.randperm(len(train_targets))   # shuffled pairs
    for start in range(0, len(order), batch_size):
        batch = order[start:start + batch_size]
        scores = model(train_contexts[batch].to(device))
        batch_targets = train_targets[batch].to(device)
        loss = F.cross_entropy(scores, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    val_perplexity = perplexity(model, val_contexts, val_targets)
    print(f"epoch {epoch + 1}: validation perplexity "
          f"{val_perplexity:.1f}")
    if val_perplexity < best_perplexity:
        best_perplexity, best_epoch = val_perplexity, epoch + 1
        best_model = copy.deepcopy(model)


# (4) Test perplexity of the best model
test_perplexity = perplexity(best_model, test_contexts, test_targets)
print(f"\ntest perplexity, epoch {best_epoch}: {test_perplexity:.1f}")


# (5) The nearest neighbors of a few words. nn.Embedding stores W
# transposed: row i is the embedding of word i. Divide each row by
# its length, so that a dot product is a cosine similarity.
embeddings = best_model[0].weight.detach().cpu()
embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)

print()
for word in ["three", "he", "city", "film", "excellent", "terrible"]:
    similarities = embeddings @ embeddings[word_index[word]]
    ranking = similarities.argsort(descending=True).tolist()
    nearest = [vocabulary[i] for i in ranking[1:6]]   # [0]: the word
    print(f"{word}: {', '.join(nearest)}")

#Exercice 5.4. Solve analogies with Glove

In [ ]:
# import GloVe vectors
import json

from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

repo = "sentence-transformers/average_word_embeddings_glove.6B.300d"
weights_path = hf_hub_download(
    repo, "0_WordEmbeddings/model.safetensors")
words_path = hf_hub_download(
    repo, "0_WordEmbeddings/whitespacetokenizer_config.json")
glove_vectors = load_file(weights_path)["emb_layer.weight"][1:]
glove_words = json.load(open(words_path))["vocab"][1:]

# Keep the 30,000 most frequent words
words = glove_words[:30000]
vectors = glove_vectors[:30000]
word_index = {word: i for i, word in enumerate(words)}

vectors = vectors / vectors.norm(dim=1, keepdim=True)

def answer(a, b, c, exclude):
  target = vectors[word_index[b]] - vectors[word_index[a]] + vectors[word_index[c]]
  similarities = vectors @ target
  if exclude:
    for word in [a, b, c]:
      similarities[word_index[word]] = -math.inf
  return words[similarities.argmax()]

questions = [("man", "king", "woman"), ("france", "paris", "spain"),
             ("big", "bigger", "small"), ("walk", "walked", "swim"),
             ("man", "doctor", "woman")]
print(f"{'question':34} excluded   allowed")
for a, b, c in questions:
    question = f"{a} is to {b} as {c} is to"
    print(f"{question:34} {answer(a, b, c, True):10} "
          f"{answer(a, b, c, False)}")


question                           excluded   allowed
man is to king as woman is to      queen      king
france is to paris as spain is to  madrid     madrid
big is to bigger as small is to    larger     larger
walk is to walked as swim is to    swam       swim
man is to doctor as woman is to    physician  doctor


#Exercice 5.5a. Start the averaging network from GloVe.

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

# train, test, val
def clean(texts):
    return [text.replace("<br />", " ") for text in texts]

split = imdb["train"].train_test_split(test_size=5000, seed=0)
train_texts = clean(split["train"]["text"])
val_texts = clean(split["test"]["text"])
test_texts = clean(imdb["test"]["text"])
train_labels = split["train"]["label"]
val_labels = split["test"]["label"]
test_labels = imdb["test"]["label"]

from sklearn.feature_extraction.text import CountVectorizer

# count the words of each review
vectorizer = CountVectorizer(max_features=10000,
                             token_pattern=r"\b\w+\b")
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

# GloVe vectors
import json

from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

repo = "sentence-transformers/average_word_embeddings_glove.6B.300d"
weights_path = hf_hub_download(
    repo, "0_WordEmbeddings/model.safetensors")
words_path = hf_hub_download(
    repo, "0_WordEmbeddings/whitespacetokenizer_config.json")
glove_vectors = load_file(weights_path)["emb_layer.weight"][1:]
glove_words = json.load(open(words_path))["vocab"][1:]

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

0_WordEmbeddings/model.safetensors: reconstructing file:   0%|          |  0.00B /  480MB            

0_WordEmbeddings/model.safetensors: downloading bytes:           |  0.00B            

whitespacetokenizer_config.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

In [ ]:
# (1) Build the pre-train matrix W (transfer learning)
import torch

glove_index = {word:i for i, word in enumerate(glove_words)}
vocabulary = vectorizer.get_feature_names_out()
D = glove_vectors.shape[1]
W_glove = torch.zeros(D, len(vocabulary))
for j, word in enumerate(vocabulary):
  if word in glove_index:
    W_glove[:, j] = glove_vectors[glove_index[word]]

# (2) Deep averaging Network
from torch import nn

D = 300 # embedding
H = 100 # hidden
V = len(vocabulary) # size vocabulary

def make_model(embedding):
  torch.manual_seed(0)
  model = nn.Sequential(
      nn.Linear(V, D, bias=False), # averaged embeddings
      nn.Linear(D, H),
      nn.ReLU(),
      nn.Linear(H, H),
      nn.ReLU(),
      nn.Linear(H, 1)
  )
  if embedding != "from scratch":
    model[0].weight.data = W_glove.clone()
  if embedding == "frozen":
    model[0].weight.requires_grad_(False)
  return model

def frequencies(X):
  counts = torch.tensor(X.toarray())
  lengths = counts.sum(dim=1, keepdim=True)
  return counts / lengths

# (3) A function that trains a model for 1,500 batches of 64
# reviews, drawn at random from the training reviews. Every 100
# batches, it measures the validation accuracy, and keeps a copy of
# the best model.
import torch.nn.functional as F
import copy

def predict(model, X):
  probabilities = []
  with torch.no_grad():
    for step in range(0, X.shape[0], 1000):
      scores = model(frequencies(X[step:step+1000]))
      probabilities.append(torch.sigmoid(scores[:, 0]))
  return torch.cat(probabilities)


def accuracy(model, X, labels):
  right = (predict(model, X ) > 0.5) == torch.tensor(labels)
  return right.float().mean().item()


def train(model, X, labels, nbr_batches=1500):
  y = torch.tensor(labels, dtype = torch.float32)
  trained = [p for p in model.parameters() if p.requires_grad]
  optimizer = torch.optim.Adam(trained, lr=1e-3)
  loss_function = nn.BCEWithLogitsLoss() # loss for binary / multi-label classification
  best_accuracy = 0
  for step in range(1, nbr_batches+1):
    batch = torch.randint(len(y), (64,)).tolist()
    scores = model(frequencies(X[batch]))[:,0]
    loss = loss_function(scores, y[batch])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
      val_accuracy = accuracy(model, X_val, val_labels)
      if val_accuracy > best_accuracy:
        best_model = copy.deepcopy(model)
        best_accuracy = val_accuracy
  return best_model

# (4) Train the three versions, and measure their test accuracy
models = {}
embeddings = ["from scratch", "frozen", "fine-tuned"]
for embedding in embeddings:
  models[embedding] = train(make_model(embedding), X_train, train_labels)
  test_accuracy = accuracy(models[embedding], X_test, test_labels)
  print(f"{embedding:12}  test accuracy {test_accuracy:.3f}")

# (5) The nearest neighbors of "good" and "bad" among the words of
# the vocabulary, in GloVe and in W after fine-tuning

def neighbors(W, word, n=5):
  W = W / W.norm(dim=0, keepdim=True)
  idx = vectorizer.vocabulary_[word]
  word_vector = W[:, idx]
  similarities = word_vector @ W
  ranking = similarities.argsort(descending=True).tolist()
  return [vocabulary[j] for j in ranking if j != idx][:n]

W_tuned = models["fine-tuned"][0].weight.detach()
for word in ["good", "bad"]:
    print(f"{word}, GloVe:      {', '.join(neighbors(W_glove, word))}")
    print(f"{word}, fine-tuned: {', '.join(neighbors(W_tuned, word))}")

from scratch  test accuracy 0.876
frozen        test accuracy 0.833
fine-tuned    test accuracy 0.881
good, GloVe:      feinstone, imho, hackenstein, hadn, sheeta
good, fine-tuned: really, always, you, think, better
bad, GloVe:      sheeta, zelah, hadn, ossessione, pazu
bad, fine-tuned: worse, unfortunately, awful, nothing, terrible


#Exercice 7.2b. Classify the IMBD reviews with an LSTM.

In [ ]:
from datasets import load_dataset
imdb = load_dataset("stanfordnlp/imdb")

# train, val, test
def clean(texts):
    return [text.replace("<br />", " ") for text in texts]

split = imdb["train"].train_test_split(test_size=5000, seed=0)
train_texts = clean(split["train"]["text"])
val_texts = clean(split["test"]["text"])
test_texts = clean(imdb["test"]["text"])
train_labels = split["train"]["label"]
val_labels = split["test"]["label"]
test_labels = imdb["test"]["label"]

hard_reviews = [
    # Negation
    ("Not bad at all, actually.", 1),
    ("Nothing about this movie is good.", 0),
    ("I wouldn't call it a masterpiece, but I didn't regret "
     "a minute.", 1),
    # Contrast
    ("Great acting, great lighting and a flabbergasting soundtrack, "
     "but not even Zendaya could save this boring, boring plot.", 0),
    ("The first hour is slow and the dialogue is clumsy, but the "
     "ending makes it all worth it.", 1),
    ("Everyone told me it would be the film of the year: great "
     "cast, amazing trailer, brilliant director. What a "
     "disappointment.", 0),
    # Sarcasm
    ("Wow, what a masterpiece. Two hours of pure brilliance, and I "
     "only checked my watch twenty times. The acting was so "
     "convincing that I truly believed the actors wanted to be "
     "somewhere else. A perfect film if you love naps.", 0),
    ("Brilliant. Another sequel nobody asked for, with the same "
     "jokes as the first one.", 0),
    # The same words in a different order
    ("I expected it to be terrible, but it was great.", 1),
    ("I expected it to be great, but it was terrible.", 0),
    # Synonyms, no word in common
    ("Excellent movie, awesome soundtrack.", 1),
    ("Superb film, wonderful score.", 1),
    # Rare words
    ("A flabbergasting, mesmerizing tour de force.", 1),
]

# The vocabulary of exercise 3.2: the 10,000 most frequent words
# of the training reviews. build_analyzer() returns the function
# that CountVectorizer uses to cut a text into words.
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=10000,
                             token_pattern=r"\b\w+\b")
vectorizer.fit(train_texts)
split_words = vectorizer.build_analyzer()
word_index = {word: int(i)   # word -> index, from 0 to V - 1
              for word, i in vectorizer.vocabulary_.items()}
V = len(word_index)

# Turn a review into the list of the indices of its words, in
# order. The words outside the vocabulary get the index V. Keep
# only the last 400 words of the long reviews.
max_length = 400

def to_indices(text):
    indices = [word_index.get(word, V) for word in split_words(text)]
    return indices[-max_length:]

train_sequences = [to_indices(text) for text in train_texts]
val_sequences = [to_indices(text) for text in val_texts]
test_sequences = [to_indices(text) for text in test_texts]

example = hard_reviews[9][0]
print(f"\n{split_words(example)}")
print(f"{to_indices(example)}")



['i', 'expected', 'it', 'to', 'be', 'great', 'but', 'it', 'was', 'terrible']
[4467, 3255, 4805, 9081, 904, 4001, 1347, 4805, 9680, 8932]


In [ ]:
# (1) The model: the embeddings of the words (one more for the
# unknown words), an LSTM that reads them in order, and a head
# that reads its last state. The reviews of a batch have different
# lengths: pad them with zeros to the same length, and pack them
# with their lengths, so that the LSTM stops at the last word of
# each review and never reads the padding.
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMClassifier(nn.Module):
  def __init__(self, V, D, H):
    super().__init__()
    # embedding attend en entrée: (B, L)
    # B: nombre de séquences dans le batch
    # L: nombre de tokens dans chaque séquence
    self.embedding = nn.Embedding(V+1, D)
    # LSTM attend en entrée: (B, L, D)
    # B: nombre de séquences dans le batch
    # L: nombre de tokens dans chaque séqence
    # D: dimension de chaque token
    self.lstm = nn.LSTM(D, H, batch_first=True)
    # head attend en entrée: (B, H)
    # B: nombre de séquences en entrée
    # H: dimension du dernier état caché
    self.head = nn.Linear(H, 1)
    # output ressort un score : (B, 1)
    # sigmoid(score) = p(positive)

  def forward(self, sequences):
    lengths = torch.tensor([len(s) for s in sequences])
    padded = nn.utils.rnn.pad_sequence([torch.tensor(s) for s in sequences], batch_first=True)
    # output: (B, L)
    x = self.embedding(padded.to(device))
    # output: (B, L, D)
    packed = nn.utils.rnn.pack_padded_sequence(
        x, lengths, batch_first=True, enforce_sorted=False
    )
    _, (h, c)= self.lstm(packed)
    # output: h: (1, B, H)
    return self.head(h[0])[:, 0]

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = LSTMClassifier(V, D=100, H=128).to(device)


# (2) Train with Adam on batches of 64 reviews, with gradient
# clipping. After each epoch, measure the validation accuracy, and
# keep a copy of the best model.
import copy
import random

def predict(model, sequences):
  probabilities = []
  with torch.no_grad():
    for step in range(0, len(sequences), 500):
      scores = model(sequences[step:step+500])
      # output: (B, 1)
      probabilities.append(torch.sigmoid(scores))
  return torch.cat(probabilities)

def accuracy(model, sequences, labels):
  right = (predict(model, sequences) > 0.5) == torch.tensor(labels)
  return right.float().mean().item()

def train(model, sequences, labels):
  y = torch.tensor(labels, dtype=torch.float32)
  optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)
  loss_function = nn.BCEWithLogitsLoss()
  batch_size = 64
  best_accuracy = -1
  for epoch in range(6):
    order = torch.randperm(len(sequences)).tolist()
    for step in range(0, len(order), batch_size):
      batch = order[step:step+batch_size]
      scores = model([sequences[i] for i in batch])
      loss = loss_function(scores, y[batch])
      optimizer.zero_grad()
      loss.backward()
      nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      optimizer.step()
    current_accuracy = accuracy(model, val_sequences, val_labels)
    print(f"epoch {epoch + 1}: "
          f"validation accuracy {current_accuracy:.3f}")
    if current_accuracy > best_accuracy:
      best_model = copy.deepcopy(model)
      best_accuracy = current_accuracy
      best_epoch = epoch +1
  return best_model, best_accuracy


# (3) The test accuracy of the best model, then the shuffle test:
# the same test reviews, with the words of each one in a random order
best_model, best_accuracy = train(model, train_sequences, train_labels)
test_accuracy = accuracy(best_model, test_sequences, test_labels)

# (4) Run the best model on the hard reviews, as in exercise 3.4
hard_sequences = [to_indices(text) for text, label in hard_reviews]
p_positive = predict(best_model, hard_sequences)

count(0)
for (review, label), proba in zip(hard_reviews, p_positive.tolist()):
  if proba > 0.5 and label == 1:
    count += 1
  if proba < 0.5 and label == 0:
    count +=1
print(count / len(hard_reviews))